In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q \
    pymupdf \
    tiktoken \
    openai \
    faiss-cpu \
    rank-bm25 \
    sentence-transformers \
    fastapi \
    uvicorn \
    pydantic \
    python-dotenv \
    pytest

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 62.6 MB/s eta 0:00:00


In [3]:
!pip uninstall -y openai
!pip install -q groq sentence-transformers faiss-cpu rank-bm25

Found existing installation: openai 2.45.0
Uninstalling openai-2.45.0:
  Successfully uninstalled openai-2.45.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.3 MB/s eta 0:00:00


In [4]:
import os
from pathlib import Path

PROJECT_ROOT = Path("/content/Python_AI_engine")
DATA_DIR = PROJECT_ROOT / "data"
PAPERS_DIR = DATA_DIR / "papers"

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
PAPERS_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_ROOT)
print("Papers:", PAPERS_DIR)

Project: /content/Python_AI_engine
Papers: /content/Python_AI_engine/data/papers


In [5]:
import os
from google.colab import userdata

GROQ_API_KEY = userdata.get("OPEN_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("Groq API key is missing from Colab Secrets.")

os.environ["GROQ_API_KEY"] = GROQ_API_KEY

print("Groq API key loaded.")

Groq API key loaded.


In [6]:
from groq import Groq

groq_client = Groq(
    api_key=os.environ["GROQ_API_KEY"]
)

print("Groq client initialized.")

Groq client initialized.


In [7]:
response = groq_client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "user",
            "content": "Explain RAG in one sentence."
        }
    ],
    temperature=0
)

answer = response.choices[0].message.content

print(answer)

RAG (Retrieve, Augment, Generate) is a type of artificial intelligence model that combines retrieval of relevant information from a database with generation of text to produce more accurate and informative responses.


In [8]:
import os

from google.colab import userdata
from groq import Groq
from sentence_transformers import SentenceTransformer


# ============================================================
# 1. Configuration
# ============================================================

EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"
GROQ_MODEL = "llama-3.3-70b-versatile"


# ============================================================
# 2. Load Groq API key from Colab Secrets
# ============================================================

GROQ_API_KEY = userdata.get("OPEN_API_KEY")

if not GROQ_API_KEY:
    raise ValueError(
        "Groq API key not found. "
        "Add your Groq API key to Colab Secrets."
    )

os.environ["GROQ_API_KEY"] = GROQ_API_KEY


# ============================================================
# 3. Initialize Groq client
# ============================================================

groq_client = Groq(
    api_key=os.environ["GROQ_API_KEY"]
)


# ============================================================
# 4. Load local embedding model
# ============================================================

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL
)


print("AI configuration loaded successfully.")
print(f"Embedding model : {EMBEDDING_MODEL}")
print(f"Embedding size  : {embedding_model.get_sentence_embedding_dimension()}")
print(f"Groq model      : {GROQ_MODEL}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

AI configuration loaded successfully.
Embedding model : BAAI/bge-small-en-v1.5
Embedding size  : 384
Groq model      : llama-3.3-70b-versatile


/tmp/ipykernel_1217/406212589.py:51: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Embedding size  : {embedding_model.get_sentence_embedding_dimension()}")


In [9]:
def embed_text(text: str):
    """
    Convert text into a normalized dense embedding vector.
    """

    return embedding_model.encode(
        text,
        normalize_embeddings=True
    )


text = "Research papers contain structured and unstructured knowledge."

vector = embed_text(text)

print("Embedding generated successfully.")
print("Vector shape:", vector.shape)

Embedding generated successfully.
Vector shape: (384,)


In [10]:
def embed_texts(texts: list[str]):
    """
    Generate normalized embeddings for multiple text chunks.
    """

    return embedding_model.encode(
        texts,
        normalize_embeddings=True,
        show_progress_bar=True
    )

texts = [
    "Graph neural networks operate on graph-structured data.",
    "Transformers are widely used for natural language processing.",
    "Research papers describe experiments and methodologies."
]

vectors = embed_texts(texts)

print("Number of embeddings:", len(vectors))
print("Embedding dimensions:", vectors.shape[1])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Number of embeddings: 3
Embedding dimensions: 384


In [11]:
def generate_answer(
    prompt: str,
    temperature: float = 0.0
) -> str:
    """
    Generate a response using the configured Groq LLM.
    """

    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=temperature
    )

    return response.choices[0].message.content

answer = generate_answer(
    "Explain Retrieval-Augmented Generation in two sentences."
)

print(answer)

Retrieval-Augmented Generation (RAG) is a natural language processing technique that combines the strengths of retrieval-based and generation-based approaches to produce more accurate and informative text outputs. By leveraging a retrieval mechanism to fetch relevant information from a large database or knowledge base, RAG models can generate more coherent and context-specific text that is grounded in existing knowledge, rather than relying solely on learned patterns and associations.


In [12]:
from dataclasses import dataclass, field


@dataclass
class DocumentPage:
    page_number: int
    text: str


@dataclass
class ResearchDocument:
    source: str
    pages: list[DocumentPage] = field(default_factory=list)

    @property
    def full_text(self) -> str:
        return "\n\n".join(
            page.text for page in self.pages
        )

    @property
    def page_count(self) -> int:
        return len(self.pages)

In [13]:
import fitz


def extract_pdf(pdf_path: str) -> ResearchDocument:
    """
    Extract text from a PDF while preserving page boundaries.
    """

    document = fitz.open(pdf_path)

    pages = []

    try:
        for page_number, page in enumerate(document, start=1):

            text = page.get_text("text").strip()

            if text:
                pages.append(
                    DocumentPage(
                        page_number=page_number,
                        text=text
                    )
                )

    finally:
        document.close()

    return ResearchDocument(
        source=pdf_path,
        pages=pages
    )

In [14]:
from google.colab import files

uploaded = files.upload()

Saving summer_intern_project_report.pdf to summer_intern_project_report.pdf


In [15]:
import shutil

filename = next(iter(uploaded.keys()))

pdf_path = PAPERS_DIR / filename

shutil.move(filename, pdf_path)

print("Paper saved to:")
print(pdf_path)

Paper saved to:
/content/Python_AI_engine/data/papers/summer_intern_project_report.pdf


In [16]:
document = extract_pdf(str(pdf_path))

print("Source:", document.source)
print("Pages extracted:", document.page_count)
print("Characters extracted:", len(document.full_text))

Source: /content/Python_AI_engine/data/papers/summer_intern_project_report.pdf
Pages extracted: 39
Characters extracted: 44627


In [17]:
for page in document.pages[:10]:
    print(
        f"Page {page.page_number}: "
        f"{len(page.text):,} characters"
    )

print(document.full_text[:5000])

Page 1: 605 characters
Page 2: 725 characters
Page 3: 998 characters
Page 4: 896 characters
Page 5: 2,552 characters
Page 6: 999 characters
Page 7: 2,067 characters
Page 8: 575 characters
Page 9: 1,904 characters
Page 10: 1,801 characters
Graph Neural Network Based Framework
for Blockchain Wallet Risk Prediction
A Project Report Submitted in complete Fulfillment of the Requirements for the Award
of the Degree of
Bachelor of Technology
in
Computer Science and Engineering
By
Pavitra Laxmi P
(N210036)
Nikhila S
(N210529)
Vijaya Lakshmi T
(N210710)
Ravi Sankar M
(N210675)
Under the Guidance of
Kumar Anurupam
Assistant Professor
Department of Computer Science and Engineering
DEPARTMENT OF COMPUTER SCIENCE AND ENGINEERING
Rajiv Gandhi University of Knowledge Technologies – Nuzvid
Nuzvid, Krishna District, Andhra Pradesh – 521202
July 2026

RAJIV GANDHI UNIVERSITY OF KNOWLEDGE TECHNOLOGIES
(A.P. Government Act 18 of 2008) RGUKT–Nuzvid, Krishna Dist – 521202
Tel: 08656-235557 / 235150
CERTIFIC

In [18]:
from dataclasses import dataclass


@dataclass
class DocumentChunk:
    chunk_id: int
    paper: str
    page_start: int
    page_end: int
    text: str
    token_count: int

In [19]:
import tiktoken

tokenizer = tiktoken.get_encoding("cl100k_base")

print("Tokenizer loaded.")

Tokenizer loaded.


In [20]:
def tokenize(text: str) -> list[int]:
    return tokenizer.encode(text)


def detokenize(tokens: list[int]) -> str:
    return tokenizer.decode(tokens)

In [21]:
sample = "Graph neural networks are useful for modeling relationships between entities."

tokens = tokenize(sample)

print("Text:", sample)
print("Token count:", len(tokens))
print("Tokens:", tokens)
print("Decoded:", detokenize(tokens))

Text: Graph neural networks are useful for modeling relationships between entities.
Token count: 11
Tokens: [11461, 30828, 14488, 527, 5505, 369, 34579, 12135, 1990, 15086, 13]
Decoded: Graph neural networks are useful for modeling relationships between entities.


In [22]:
def create_chunks(
    document: ResearchDocument,
    chunk_size: int = 400,
    overlap: int = 80
) -> list[DocumentChunk]:

    if overlap >= chunk_size:
        raise ValueError(
            "overlap must be smaller than chunk_size"
        )

    chunks = []
    chunk_id = 0

    for page in document.pages:

        tokens = tokenize(page.text)

        start = 0

        while start < len(tokens):

            end = min(
                start + chunk_size,
                len(tokens)
            )

            chunk_tokens = tokens[start:end]

            text = detokenize(chunk_tokens).strip()

            if text:
                chunks.append(
                    DocumentChunk(
                        chunk_id=chunk_id,
                        paper=document.source,
                        page_start=page.page_number,
                        page_end=page.page_number,
                        text=text,
                        token_count=len(chunk_tokens)
                    )
                )

                chunk_id += 1

            # Move forward while preserving overlap
            start += chunk_size - overlap

    return chunks

In [23]:
chunks = create_chunks(
    document,
    chunk_size=400,
    overlap=80
)

print("Total chunks:", len(chunks))

Total chunks: 51


In [24]:
for chunk in chunks[:5]:
    print("=" * 80)
    print("Chunk ID:", chunk.chunk_id)
    print("Page:", chunk.page_start)
    print("Tokens:", chunk.token_count)
    print(chunk.text[:1000])

Chunk ID: 0
Page: 1
Tokens: 154
Graph Neural Network Based Framework
for Blockchain Wallet Risk Prediction
A Project Report Submitted in complete Fulfillment of the Requirements for the Award
of the Degree of
Bachelor of Technology
in
Computer Science and Engineering
By
Pavitra Laxmi P
(N210036)
Nikhila S
(N210529)
Vijaya Lakshmi T
(N210710)
Ravi Sankar M
(N210675)
Under the Guidance of
Kumar Anurupam
Assistant Professor
Department of Computer Science and Engineering
DEPARTMENT OF COMPUTER SCIENCE AND ENGINEERING
Rajiv Gandhi University of Knowledge Technologies – Nuzvid
Nuzvid, Krishna District, Andhra Pradesh – 521202
July 2026
Chunk ID: 1
Page: 2
Tokens: 214
RAJIV GANDHI UNIVERSITY OF KNOWLEDGE TECHNOLOGIES
(A.P. Government Act 18 of 2008) RGUKT–Nuzvid, Krishna Dist – 521202
Tel: 08656-235557 / 235150
CERTIFICATE OF COMPLETION
This is to certify that the work entitled Graph Neural Network
Based Framework for Blockchain Wallet Risk Prediction is the
bonafide work of Pavitra Laxmi P (

In [25]:
token_counts = [chunk.token_count for chunk in chunks]

print("Number of chunks:", len(token_counts))
print("Minimum tokens:", min(token_counts))
print("Maximum tokens:", max(token_counts))
print("Average tokens:", sum(token_counts) / len(token_counts))

Number of chunks: 51
Minimum tokens: 14
Maximum tokens: 400
Average tokens: 223.19607843137254


In [26]:
chunk = chunks[5]

print("Chunk ID:", chunk.chunk_id)
print("Paper:", chunk.paper)
print("Page:", chunk.page_start)
print("Token count:", chunk.token_count)
print()
print(chunk.text)

Chunk ID: 5
Paper: /content/Python_AI_engine/data/papers/summer_intern_project_report.pdf
Page: 5
Token count: 400

. . . . . . . . . . . . . . . . . . . . . .10
2.3 Functional Requirements . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .11
2.4 Non-Functional Requirements . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .11
CHAPTER 3
SYSTEM DESIGN
3.1 Environment Setup . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 13
3.2 Dataset . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .14
3.3 Feature Engineering . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 15
3.4 Model Training . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 16
3.5 Model Predictions . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 17
3.6 Model Evaluation . . . . . . . . . . . . . . . . . . . . 

In [27]:
chunk_texts = [chunk.text for chunk in chunks]

chunk_embeddings = embed_texts(chunk_texts)

print("Chunks:", len(chunks))
print("Embeddings shape:", chunk_embeddings.shape)

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Chunks: 51
Embeddings shape: (51, 384)


In [28]:
print("First embedding:")
print(chunk_embeddings[0])

print("\nShape:")
print(chunk_embeddings[0].shape)

First embedding:
[-2.99231596e-02 -1.51126077e-02 -9.37252939e-02 -1.36942239e-02
  1.58576407e-02  1.25433747e-02  2.53992788e-02  2.40227748e-02
  3.83261032e-02  2.32176241e-02  3.35699059e-02 -8.88738036e-03
  1.68864802e-02  6.16238751e-02  9.61974822e-03 -4.74155648e-03
  1.52489718e-03  4.16343473e-03  4.23817560e-02  4.69949730e-02
  5.51118702e-02 -1.19686676e-02 -2.07974259e-02 -1.01074979e-01
  2.79451348e-02  4.61348146e-02  3.66600528e-02 -5.37638515e-02
 -5.13546318e-02 -1.94872633e-01  6.51267022e-02 -4.03689072e-02
  8.19389373e-02 -3.06664538e-02  3.32126208e-02  1.44662634e-02
  1.39402058e-02  3.24993394e-02 -2.88448995e-03  2.24914812e-02
 -4.02692854e-02  9.26891156e-03  3.14761624e-02 -5.03061479e-03
  7.57333785e-02 -6.33822903e-02 -1.41809611e-02 -3.43014039e-02
 -6.65147156e-02 -2.00069640e-02 -1.72596052e-02 -3.02268751e-02
 -1.05984192e-02  2.43812595e-02  3.33331525e-02 -6.16656151e-03
  1.13584111e-02  3.88136506e-02 -4.69439402e-02  3.10849193e-02
  1.2519

In [29]:
import faiss
import numpy as np

In [30]:
embedding_dimension = chunk_embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(
    embedding_dimension
)

faiss_index.add(
    np.asarray(chunk_embeddings, dtype="float32")
)

print("FAISS index created.")
print("Vectors indexed:", faiss_index.ntotal)

FAISS index created.
Vectors indexed: 51


In [31]:
def semantic_search(
    query: str,
    top_k: int = 5
):
    """
    Retrieve the most semantically similar document chunks.
    """

    query_embedding = embed_text(query)

    query_vector = np.asarray(
        [query_embedding],
        dtype="float32"
    )

    scores, indices = faiss_index.search(
        query_vector,
        top_k
    )

    results = []

    for score, index in zip(scores[0], indices[0]):

        if index == -1:
            continue

        chunk = chunks[index]

        results.append({
            "chunk_id": chunk.chunk_id,
            "paper": chunk.paper,
            "page": chunk.page_start,
            "score": float(score),
            "text": chunk.text
        })

    return results

In [32]:
results = semantic_search(
    "How are graph neural networks used for blockchain wallet risk prediction?",
    top_k=5
)

In [33]:
for result in results:
    print("=" * 80)
    print("Chunk:", result["chunk_id"])
    print("Page:", result["page"])
    print("Score:", result["score"])
    print(result["text"][:1000])

Chunk: 10
Page: 7
Score: 0.9137974381446838
ABSTRACT
The rapid growth of cryptocurrency transactions has made blockchain net-
works an attractive channel for illicit activity, including money laundering,
fraud, and ransomware payments. Identifying high-risk wallets from raw
transaction data is difficult because risk is not encoded in any single at-
tribute of a wallet but emerges from the pattern of transactions and the
surrounding network of counterparties.
This project proposes a Graph
Neural Network based Wallet Risk Prediction system that models the Bit-
coin transaction network as a graph, where wallet addresses form nodes
and transactions between them form edges.
The system follows a multi-stage pipeline. Raw wallet-level features and
the address-to-address transaction edge list are first combined through a
Flow Dynamics Representation Module (FDRM), which engineers struc-
tural, transactional, behavioural, temporal, and connectivity features for
every wallet, including a composi

In [34]:
query1 = "What dataset was used for blockchain wallet risk prediction?"

query2 = "What model architecture was used?"

In [35]:
for query in [query1, query2]:

    print("\n" + "#" * 100)
    print("QUERY:", query)

    results = semantic_search(query, top_k=3)

    for result in results:
        print("-" * 80)
        print(
            f"Chunk={result['chunk_id']} "
            f"Page={result['page']} "
            f"Score={result['score']:.4f}"
        )
        print(result["text"][:500])


####################################################################################################
QUERY: What dataset was used for blockchain wallet risk prediction?
--------------------------------------------------------------------------------
Chunk=10 Page=7 Score=0.8421
ABSTRACT
The rapid growth of cryptocurrency transactions has made blockchain net-
works an attractive channel for illicit activity, including money laundering,
fraud, and ransomware payments. Identifying high-risk wallets from raw
transaction data is difficult because risk is not encoded in any single at-
tribute of a wallet but emerges from the pattern of transactions and the
surrounding network of counterparties.
This project proposes a Graph
Neural Network based Wallet Risk Prediction system 
--------------------------------------------------------------------------------
Chunk=15 Page=10 Score=0.8395
1.2 OBJECTIVES
Objectives of the Proposed Wallet Risk Prediction System
• Transaction Graph Construction: To

In [36]:
from rank_bm25 import BM25Okapi
import re

In [37]:
def tokenize_for_bm25(text: str) -> list[str]:
    """
    Simple lexical tokenizer for BM25.
    Converts text to lowercase and extracts word-like tokens.
    """
    return re.findall(r"\b\w+\b", text.lower())

In [38]:
sample = "Graph Neural Networks (GNNs) are used for wallet-risk prediction."

print(tokenize_for_bm25(sample))

['graph', 'neural', 'networks', 'gnns', 'are', 'used', 'for', 'wallet', 'risk', 'prediction']


In [39]:
bm25_corpus = [
    tokenize_for_bm25(chunk.text)
    for chunk in chunks
]

bm25_index = BM25Okapi(bm25_corpus)

print("BM25 index created.")
print("Documents indexed:", len(bm25_corpus))

BM25 index created.
Documents indexed: 51


In [40]:
def bm25_search(
    query: str,
    top_k: int = 5
):
    """
    Retrieve chunks using lexical BM25 ranking.
    """

    query_tokens = tokenize_for_bm25(query)

    scores = bm25_index.get_scores(query_tokens)

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for index in top_indices:

        chunk = chunks[index]

        results.append({
            "chunk_id": chunk.chunk_id,
            "paper": chunk.paper,
            "page": chunk.page_start,
            "score": float(scores[index]),
            "text": chunk.text
        })

    return results

In [41]:
results = bm25_search(
    "Graph Neural Network blockchain wallet risk prediction",
    top_k=5
)

In [42]:
for result in results:
    print("=" * 80)
    print("Chunk:", result["chunk_id"])
    print("Page:", result["page"])
    print("BM25 Score:", result["score"])
    print(result["text"][:700])

Chunk: 46
Page: 36
BM25 Score: 7.3531749640117985
CHAPTER 6
EXPLAINABILITY
6.1 Explainability Overview
The Blockchain Wallet Risk Prediction framework incorporates an ex-
plainability module to improve the transparency and interpretability of
Graph Neural Network predictions. Since blockchain forensic investiga-
tions require analysts to justify why a wallet has been classified as risky,
the system combines both global and local explanation techniques. Global
explanations identify the most influential engineered features across the
entire dataset, while local explanations describe the specific transaction
neighbourhood and node features responsible for an individual wallet pre-
diction.
6.2 Global Feature Importance
Permutation Importance i
Chunk: 13
Page: 9
BM25 Score: 7.3211008388868075
CHAPTER 1
1.1 INTRODUCTION
Blockchain platforms such as Bitcoin record every transaction on a
public, immutable ledger, generating vast amounts of transactional data.
While this transparency is a core

In [43]:
query = "What model was used for blockchain wallet risk prediction?"

In [44]:
dense_results = semantic_search(
    query,
    top_k=5
)

print("FAISS RESULTS")

for result in dense_results:
    print(
        f"Chunk={result['chunk_id']} "
        f"Page={result['page']} "
        f"Score={result['score']:.4f}"
    )


keyword_results = bm25_search(
    query,
    top_k=5
)

print("\nBM25 RESULTS")

for result in keyword_results:
    print(
        f"Chunk={result['chunk_id']} "
        f"Page={result['page']} "
        f"Score={result['score']:.4f}"
    )

FAISS RESULTS
Chunk=10 Page=7 Score=0.8631
Chunk=15 Page=10 Score=0.8551
Chunk=48 Page=38 Score=0.8486
Chunk=13 Page=9 Score=0.8449
Chunk=0 Page=1 Score=0.8388

BM25 RESULTS
Chunk=28 Page=21 Score=7.5456
Chunk=24 Page=18 Score=6.9500
Chunk=46 Page=36 Score=6.8226
Chunk=23 Page=17 Score=6.5431
Chunk=48 Page=38 Score=6.2397


In [45]:
def reciprocal_rank_fusion(
    result_lists: list[list[dict]],
    k: int = 60,
    top_k: int = 5
):
    """
    Combine ranked retrieval results using
    Reciprocal Rank Fusion (RRF).
    """

    fused_scores = {}
    result_lookup = {}

    for results in result_lists:

        for rank, result in enumerate(results, start=1):

            chunk_id = result["chunk_id"]

            fused_scores[chunk_id] = (
                fused_scores.get(chunk_id, 0.0)
                + 1.0 / (k + rank)
            )

            result_lookup[chunk_id] = result

    ranked_chunks = sorted(
        fused_scores.items(),
        key=lambda item: item[1],
        reverse=True
    )

    final_results = []

    for chunk_id, fusion_score in ranked_chunks[:top_k]:

        result = result_lookup[chunk_id].copy()

        result["fusion_score"] = fusion_score

        final_results.append(result)

    return final_results

In [46]:
def hybrid_search(
    query: str,
    top_k: int = 5,
    candidate_k: int = 10
):
    """
    Combine dense FAISS retrieval and BM25
    retrieval using Reciprocal Rank Fusion.
    """

    dense_results = semantic_search(
        query,
        top_k=candidate_k
    )

    keyword_results = bm25_search(
        query,
        top_k=candidate_k
    )

    return reciprocal_rank_fusion(
        [dense_results, keyword_results],
        top_k=top_k
    )

In [47]:
query = "What model architecture was used for blockchain wallet risk prediction?"

results = hybrid_search(
    query,
    top_k=5
)

for result in results:

    print("=" * 80)

    print(
        f"Chunk: {result['chunk_id']} | "
        f"Page: {result['page']} | "
        f"Fusion: {result['fusion_score']:.5f}"
    )

    print(result["text"][:1000])

Chunk: 10 | Page: 7 | Fusion: 0.03279
ABSTRACT
The rapid growth of cryptocurrency transactions has made blockchain net-
works an attractive channel for illicit activity, including money laundering,
fraud, and ransomware payments. Identifying high-risk wallets from raw
transaction data is difficult because risk is not encoded in any single at-
tribute of a wallet but emerges from the pattern of transactions and the
surrounding network of counterparties.
This project proposes a Graph
Neural Network based Wallet Risk Prediction system that models the Bit-
coin transaction network as a graph, where wallet addresses form nodes
and transactions between them form edges.
The system follows a multi-stage pipeline. Raw wallet-level features and
the address-to-address transaction edge list are first combined through a
Flow Dynamics Representation Module (FDRM), which engineers struc-
tural, transactional, behavioural, temporal, and connectivity features for
every wallet, including a composite Flo

In [48]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

print("Reranker loaded.")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Reranker loaded.


In [49]:
query = "What model architecture was used for blockchain wallet risk prediction?"

candidates = hybrid_search(
    query=query,
    top_k=10,
    candidate_k=10
)

print("Candidates retrieved:", len(candidates))

print("\nFirst candidate keys:")
print(candidates[0].keys())

Candidates retrieved: 10

First candidate keys:
dict_keys(['chunk_id', 'paper', 'page', 'score', 'text', 'fusion_score'])


In [50]:
test_pair = [
    (
        query,
        candidates[0]["text"]
    )
]

score = reranker.predict(test_pair)

print("Reranker score:", float(score[0]))

Reranker score: 5.792764663696289


In [51]:
def rerank_results(
    query: str,
    candidates: list[dict],
    top_k: int = 5
):
    if not candidates:
        return []

    pairs = [
        (query, candidate["text"])
        for candidate in candidates
    ]

    scores = reranker.predict(pairs)

    reranked = []

    for candidate, score in zip(candidates, scores):
        result = candidate.copy()
        result["rerank_score"] = float(score)
        reranked.append(result)

    reranked.sort(
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    return reranked[:top_k]

In [52]:
reranked_results = rerank_results(
    query=query,
    candidates=candidates,
    top_k=5
)

for result in reranked_results:
    print("=" * 80)
    print(
        f"Chunk: {result['chunk_id']} | "
        f"Page: {result['page']} | "
        f"RRF: {result['fusion_score']:.5f} | "
        f"Rerank: {result['rerank_score']:.5f}"
    )
    print(result["text"][:700])

Chunk: 24 | Page: 18 | RRF: 0.01587 | Rerank: 6.55226
3. System Design
3.1 Environment Setup
We have used the Google Colab environment for the development of
the Blockchain Wallet Risk Prediction system, with datasets and model
checkpoints stored on Google Drive. All the required libraries are installed.
• numpy
• pandas
• torch
• torch.nn / torch.nn.functional
• torch geometric.data (Data)
• torch geometric.loader (NeighborLoader)
• torch geometric.nn (SAGEConv, GCNConv, GATConv)
• torch geometric.utils (coalesce)
• sklearn.model selection
• sklearn.preprocessing (StandardScaler)
• sklearn.metrics
• matplotlib.pyplot
• os, random, warnings, shutil
3.2 Dataset Description
The dataset used in this project consists of Bitcoin wallet-level feature
Chunk: 48 | Page: 38 | RRF: 0.03055 | Rerank: 5.91028
CHAPTER 7
CONCLUSION
• This project presents a Graph Neural Network based framework for
blockchain wallet risk prediction by modelling the Bitcoin transac-
tion network as a graph, allowing b

In [53]:
def build_context(results: list[dict]) -> str:
    """
    Build a context block from retrieved evidence.
    Each chunk retains its source page.
    """

    context_parts = []

    for i, result in enumerate(results, start=1):

        context_parts.append(
            f"""[Source {i}]
Paper: {result['paper']}
Page: {result['page']}
Chunk ID: {result['chunk_id']}

{result['text']}
"""
        )

    return "\n\n".join(context_parts)

In [54]:
def retrieve(
    query: str,
    candidate_k: int = 10,
    top_k: int = 5
):
    """
    Full retrieval pipeline:

    FAISS + BM25
        ↓
    Reciprocal Rank Fusion
        ↓
    Cross-encoder reranking
        ↓
    Top-K evidence
    """

    candidates = hybrid_search(
        query=query,
        top_k=candidate_k,
        candidate_k=candidate_k
    )

    return rerank_results(
        query=query,
        candidates=candidates,
        top_k=top_k
    )

In [55]:
query = "What model architecture was used for blockchain wallet risk prediction?"

results = retrieve(
    query=query,
    candidate_k=10,
    top_k=5
)

context = build_context(results)

print(context)

[Source 1]
Paper: /content/Python_AI_engine/data/papers/summer_intern_project_report.pdf
Page: 18
Chunk ID: 24

3. System Design
3.1 Environment Setup
We have used the Google Colab environment for the development of
the Blockchain Wallet Risk Prediction system, with datasets and model
checkpoints stored on Google Drive. All the required libraries are installed.
• numpy
• pandas
• torch
• torch.nn / torch.nn.functional
• torch geometric.data (Data)
• torch geometric.loader (NeighborLoader)
• torch geometric.nn (SAGEConv, GCNConv, GATConv)
• torch geometric.utils (coalesce)
• sklearn.model selection
• sklearn.preprocessing (StandardScaler)
• sklearn.metrics
• matplotlib.pyplot
• os, random, warnings, shutil
3.2 Dataset Description
The dataset used in this project consists of Bitcoin wallet-level features,
wallet risk-class labels, and an address-to-address transaction edge list, to-
gether describing both the individual behaviour of wallets and the network
of transactions that connects t

In [56]:
def build_rag_prompt(query: str, context: str) -> str:

    return f"""
You are a research assistant.

Answer the user's question using ONLY the research
evidence provided below.

Rules:
1. Do not invent facts.
2. Do not use information that is not supported by
   the provided evidence.
3. If the evidence is insufficient, say so explicitly.
4. Cite supporting evidence using [Source N].
5. Give a concise and technically accurate answer.

RESEARCH EVIDENCE
=================
{context}

QUESTION
========
{query}

ANSWER
======
"""

In [57]:
prompt = build_rag_prompt(
    query=query,
    context=context
)

answer = generate_answer(prompt)

print(answer)

The model architecture used for blockchain wallet risk prediction was Adaptive Risk-Aware GraphSAGE [Source 2]. This architecture was selected after a comparative evaluation with GCN and GAT, which demonstrated that GraphSAGE consistently achieved the best predictive performance [Source 2].


In [58]:
def rag_query(
    query: str,
    candidate_k: int = 10,
    top_k: int = 5
):
    # Retrieve evidence
    results = retrieve(
        query=query,
        candidate_k=candidate_k,
        top_k=top_k
    )

    # Build context
    context = build_context(results)

    # Build grounded prompt
    prompt = build_rag_prompt(
        query=query,
        context=context
    )

    # Generate answer
    answer = generate_answer(prompt)

    return {
        "query": query,
        "answer": answer,
        "sources": results
    }

In [59]:
result = rag_query(
    "What model architecture was used for blockchain wallet risk prediction?"
)

print("ANSWER")
print("=" * 80)
print(result["answer"])

ANSWER
The model architecture used for blockchain wallet risk prediction was Adaptive Risk-Aware GraphSAGE [Source 2]. This architecture was selected after a comparative evaluation with GCN and GAT, which demonstrated that GraphSAGE consistently achieved the best predictive performance [Source 2].


In [60]:
print("\nSOURCES")
print("=" * 80)

for source in result["sources"]:
    print(
        f"Page: {source['page']} | "
        f"Chunk: {source['chunk_id']} | "
        f"Rerank score: {source['rerank_score']:.4f}"
    )


SOURCES
Page: 18 | Chunk: 24 | Rerank score: 6.5523
Page: 38 | Chunk: 48 | Rerank score: 5.9103
Page: 7 | Chunk: 10 | Rerank score: 5.7928
Page: 36 | Chunk: 46 | Rerank score: 5.6480
Page: 10 | Chunk: 15 | Rerank score: 5.4423


In [61]:
result = rag_query(
    "What dataset was used for blockchain wallet risk prediction?"
)

print(result["answer"])

result = rag_query(
    "How was the GAT model trained and evaluated?"
)

print(result["answer"])

result = rag_query(
    "What was the total financial cost of developing this project?"
)

print(result["answer"])

The dataset used for blockchain wallet risk prediction consists of Bitcoin wallet-level features, wallet risk-class labels, and an address-to-address transaction edge list, describing both the individual behaviour of wallets and the network of transactions that connects them [Source 1]. The specific files included in the dataset are "wallets features binary.csv", which contains per-wallet, per-timestep transactional features [Source 1].
The research evidence does not provide detailed information on how the GAT model was trained and evaluated, separately from the other models. However, it is mentioned that GAT, along with GCN, was used as a baseline and was trained and evaluated under identical data splits and procedure as GraphSAGE [Source 1]. The training procedure for GraphSAGE is described in [Source 3], but it is not explicitly stated if the same procedure was used for GAT. Therefore, the evidence is insufficient to provide a detailed answer on how the GAT model was trained and eva

# External Research Discovery

Discover related research papers from scholarly sources
such as OpenAlex and arXiv, rank them using semantic
similarity and cross-encoder reranking, and provide
direct links and metadata.

In [65]:
!pip install -q requests feedparser

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 3.0 MB/s eta 0:00:00


In [66]:
import json
import requests
import feedparser
from urllib.parse import quote


def build_paper_profile(document: ResearchDocument) -> str:
    """
    Create a compact representation of the research paper
    for research discovery.
    """

    pages = document.pages[:5]

    return "\n\n".join(
        page.text
        for page in pages
    )
paper_profile = build_paper_profile(document)

print(paper_profile[:5000])

Graph Neural Network Based Framework
for Blockchain Wallet Risk Prediction
A Project Report Submitted in complete Fulfillment of the Requirements for the Award
of the Degree of
Bachelor of Technology
in
Computer Science and Engineering
By
Pavitra Laxmi P
(N210036)
Nikhila S
(N210529)
Vijaya Lakshmi T
(N210710)
Ravi Sankar M
(N210675)
Under the Guidance of
Kumar Anurupam
Assistant Professor
Department of Computer Science and Engineering
DEPARTMENT OF COMPUTER SCIENCE AND ENGINEERING
Rajiv Gandhi University of Knowledge Technologies – Nuzvid
Nuzvid, Krishna District, Andhra Pradesh – 521202
July 2026

RAJIV GANDHI UNIVERSITY OF KNOWLEDGE TECHNOLOGIES
(A.P. Government Act 18 of 2008) RGUKT–Nuzvid, Krishna Dist – 521202
Tel: 08656-235557 / 235150
CERTIFICATE OF COMPLETION
This is to certify that the work entitled Graph Neural Network
Based Framework for Blockchain Wallet Risk Prediction is the
bonafide work of Pavitra Laxmi P (N210036), Nikhila S (N210529),
Vijaya Lakshmi T (N210710) and R

In [67]:
def generate_research_queries(
    paper_profile: str,
    num_queries: int = 3
) -> list[str]:

    prompt = f"""
You are a research discovery assistant.

Analyze the research paper below and generate
{num_queries} precise scholarly search queries
for finding closely related research papers.

Focus on:
- research problem
- technical methodology
- algorithms/models
- application domain
- important technical terminology

Avoid:
- generic queries
- university names
- author names
- administrative information

Return ONLY valid JSON:

{{
    "queries": [
        "query 1",
        "query 2",
        "query 3"
    ]
}}

Research paper:
----------------
{paper_profile[:12000]}
----------------
"""

    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    content = response.choices[0].message.content.strip()

    # Handle accidental Markdown code fences
    if content.startswith("```"):
        content = content.replace("```json", "")
        content = content.replace("```", "")
        content = content.strip()

    data = json.loads(content)

    return data["queries"]

In [68]:
research_queries = generate_research_queries(
    paper_profile
)

print("Generated research queries:\n")

for i, query in enumerate(research_queries, start=1):

    print(f"{i}. {query}")

Generated research queries:

1. Graph Neural Network for blockchain risk assessment
2. Blockchain wallet risk prediction using machine learning algorithms
3. Deep learning approaches for cryptocurrency wallet security analysis


In [69]:
def extract_openalex_abstract(work: dict) -> str:
    """
    Reconstruct an abstract from OpenAlex's
    inverted-index representation.
    """

    inverted_index = work.get("abstract_inverted_index")

    if not inverted_index:
        return ""

    words = []

    for word, positions in inverted_index.items():
        for position in positions:
            words.append((position, word))

    words.sort(key=lambda x: x[0])

    return " ".join(
        word for _, word in words
    )

In [70]:
def search_openalex(
    query: str,
    max_results: int = 10
) -> list[dict]:

    url = "https://api.openalex.org/works"

    params = {
        "search": query,
        "per-page": max_results
    }

    response = requests.get(
        url,
        params=params,
        timeout=30
    )

    response.raise_for_status()

    data = response.json()

    papers = []

    for work in data.get("results", []):

        primary_location = work.get(
            "primary_location"
        ) or {}

        papers.append({
            "title": work.get("display_name", ""),
            "authors": [
                author["author"]["display_name"]
                for author in work.get("authorships", [])
                if author.get("author")
            ],
            "year": work.get("publication_year"),
            "abstract": extract_openalex_abstract(work),
            "doi": work.get("doi"),
            "url": primary_location.get(
                "landing_page_url"
            ),
            "source": "OpenAlex"
        })

    return papers

In [71]:
openalex_results = search_openalex(
    research_queries[0],
    max_results=5
)

print("OpenAlex results:", len(openalex_results))

for i, paper in enumerate(openalex_results, start=1):

    print("\n" + "=" * 80)
    print(f"#{i}")
    print("Title:", paper["title"])
    print("Authors:", ", ".join(paper["authors"][:5]))
    print("Year:", paper["year"])
    print("URL:", paper["url"])

OpenAlex results: 5

#1
Title: Blockchain for 5G and beyond networks: A state of the art survey
Authors: Dinh C. Nguyen, Pubudu N. Pathirana, Ming Ding, Aruna Seneviratne
Year: 2020
URL: https://doi.org/10.1016/j.jnca.2020.102693

#2
Title: Blockchain technology applications in healthcare: An overview
Authors: Abid Haleem, Mohd Javaid, Ravi Pratap Singh, Rajiv Suman, Shanay Rab
Year: 2021
URL: https://doi.org/10.1016/j.ijin.2021.09.005

#3
Title: A blockchain future for internet of things security: a position paper
Authors: Mandrita Banerjee, Jung­hee Lee, Kim‐Kwang Raymond Choo
Year: 2017
URL: https://doi.org/10.1016/j.dcan.2017.10.006

#4
Title: Blockchain Security Attacks, Challenges, and Solutions for the Future Distributed IoT Network
Authors: Saurabh Singh, A. S. M. Sanwar Hosen, Byungun Yoon
Year: 2021
URL: https://doi.org/10.1109/access.2021.3051602

#5
Title: An Industrial IoT-Based Blockchain-Enabled Secure Searchable Encryption Approach for Healthcare Systems Using Neural Ne

In [72]:
def search_arxiv(
    query: str,
    max_results: int = 10
) -> list[dict]:

    encoded_query = quote(query)

    url = (
        "https://export.arxiv.org/api/query"
        f"?search_query=all:{encoded_query}"
        f"&start=0"
        f"&max_results={max_results}"
        f"&sortBy=relevance"
        f"&sortOrder=descending"
    )

    response = requests.get(
        url,
        timeout=30
    )

    response.raise_for_status()

    feed = feedparser.parse(
        response.text
    )

    papers = []

    for entry in feed.entries:

        papers.append({
            "title": entry.get(
                "title", ""
            ).replace("\n", " ").strip(),

            "authors": [
                author.name
                for author in entry.get(
                    "authors", []
                )
            ],

            "year": entry.get(
                "published", ""
            )[:4],

            "abstract": entry.get(
                "summary", ""
            ).strip(),

            "doi": None,

            "url": entry.get(
                "link"
            ),

            "source": "arXiv"
        })

    return papers

In [73]:
arxiv_results = search_arxiv(
    research_queries[0],
    max_results=5
)

print("arXiv results:", len(arxiv_results))

for i, paper in enumerate(arxiv_results, start=1):

    print("\n" + "=" * 80)
    print(f"#{i}")
    print("Title:", paper["title"])
    print("Authors:", ", ".join(paper["authors"][:5]))
    print("Year:", paper["year"])
    print("URL:", paper["url"])

arXiv results: 5

#1
Title: A Tutorial about Random Neural Networks in Supervised Learning
Authors: Sebastián Basterrech, Gerardo Rubino
Year: 2016
URL: https://arxiv.org/abs/1609.04846v1

#2
Title: Sequential Design and Spatial Modeling for Portfolio Tail Risk Measurement
Authors: Michael Ludkovski, James Risk
Year: 2017
URL: https://arxiv.org/abs/1710.05204v2

#3
Title: ChainSplitter: Towards Blockchain-based Industrial IoT Architecture for Supporting Hierarchical Storage
Authors: Gang Wang, Zhijie Jerry Shi, Mark Nixon, Song Han
Year: 2019
URL: https://arxiv.org/abs/1910.00742v1

#4
Title: BlockSim: An Extensible Simulation Tool for Blockchain Systems
Authors: Maher Alharby, Aad van Moorsel
Year: 2020
URL: https://arxiv.org/abs/2004.13438v2

#5
Title: Learning Universal Graph Neural Network Embeddings With Aid Of Transfer Learning
Authors: Saurabh Verma, Zhi-Li Zhang
Year: 2019
URL: https://arxiv.org/abs/1909.10086v3


In [74]:
def discover_research(
    queries: list[str],
    results_per_query: int = 5
) -> list[dict]:

    all_papers = []

    for query in queries:

        print(f"\nSearching: {query}")

        openalex_results = search_openalex(
            query,
            max_results=results_per_query
        )

        arxiv_results = search_arxiv(
            query,
            max_results=results_per_query
        )

        all_papers.extend(openalex_results)
        all_papers.extend(arxiv_results)

    return all_papers

In [75]:
external_candidates = discover_research(
    research_queries,
    results_per_query=5
)

print(
    "\nTotal external candidates:",
    len(external_candidates)
)


Searching: Graph Neural Network for blockchain risk assessment

Searching: Blockchain wallet risk prediction using machine learning algorithms

Searching: Deep learning approaches for cryptocurrency wallet security analysis

Total external candidates: 30


In [76]:
def deduplicate_papers(
    papers: list[dict]
) -> list[dict]:

    seen_titles = set()
    unique_papers = []

    for paper in papers:

        title = paper["title"].lower().strip()

        if not title:
            continue

        if title in seen_titles:
            continue

        seen_titles.add(title)

        unique_papers.append(paper)

    return unique_papers


external_candidates = deduplicate_papers(
    external_candidates
)

print(
    "Unique external papers:",
    len(external_candidates)
)

Unique external papers: 29


In [77]:
for i, paper in enumerate(
    external_candidates,
    start=1
):

    print("\n" + "=" * 100)
    print(f"#{i}")
    print("Title:", paper["title"])
    print("Year:", paper["year"])
    print("Source:", paper["source"])
    print("URL:", paper["url"])


#1
Title: Blockchain for 5G and beyond networks: A state of the art survey
Year: 2020
Source: OpenAlex
URL: https://doi.org/10.1016/j.jnca.2020.102693

#2
Title: Blockchain technology applications in healthcare: An overview
Year: 2021
Source: OpenAlex
URL: https://doi.org/10.1016/j.ijin.2021.09.005

#3
Title: A blockchain future for internet of things security: a position paper
Year: 2017
Source: OpenAlex
URL: https://doi.org/10.1016/j.dcan.2017.10.006

#4
Title: Blockchain Security Attacks, Challenges, and Solutions for the Future Distributed IoT Network
Year: 2021
Source: OpenAlex
URL: https://doi.org/10.1109/access.2021.3051602

#5
Title: An Industrial IoT-Based Blockchain-Enabled Secure Searchable Encryption Approach for Healthcare Systems Using Neural Network
Year: 2022
Source: OpenAlex
URL: https://doi.org/10.3390/s22020572

#6
Title: A Tutorial about Random Neural Networks in Supervised Learning
Year: 2016
Source: arXiv
URL: https://arxiv.org/abs/1609.04846v1

#7
Title: Sequent

In [78]:
source_paper_text = paper_profile[:12000]

source_paper_embedding = embed_text(
    source_paper_text
)

print("Source embedding shape:", source_paper_embedding.shape)

Source embedding shape: (384,)


In [80]:
def build_external_paper_text(paper: dict) -> str:
    title = paper.get("title", "").strip()
    abstract = paper.get("abstract", "").strip()

    return (
        f"Title: {title}\n"
        f"Abstract: {abstract}"
    )


rankable_papers = [
    paper
    for paper in external_candidates
    if paper.get("title")
    and paper.get("abstract")
    and len(paper["abstract"].strip()) > 50
]

print("Papers with usable abstracts:", len(rankable_papers))

Papers with usable abstracts: 28


In [81]:
candidate_texts = [
    build_external_paper_text(paper)
    for paper in rankable_papers
]

candidate_embeddings = embed_texts(
    candidate_texts
)

print("Candidate embeddings:", candidate_embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Candidate embeddings: (28, 384)


In [82]:
similarity_scores = (
    candidate_embeddings @ source_paper_embedding
)

for paper, score in zip(
    rankable_papers,
    similarity_scores
):
    paper["embedding_similarity"] = float(score)

rankable_papers.sort(
    key=lambda paper: paper["embedding_similarity"],
    reverse=True
)

In [83]:
for i, paper in enumerate(
    rankable_papers[:10],
    start=1
):
    print("\n" + "=" * 100)
    print(f"#{i}")
    print("Title:", paper["title"])
    print("Similarity:", round(
        paper["embedding_similarity"], 4
    ))
    print("Year:", paper["year"])
    print("Source:", paper["source"])
    print("URL:", paper["url"])


#1
Title: Blockchain meets machine learning: a survey
Similarity: 0.8182
Year: 2024
Source: OpenAlex
URL: https://doi.org/10.1186/s40537-023-00852-y

#2
Title: An Industrial IoT-Based Blockchain-Enabled Secure Searchable Encryption Approach for Healthcare Systems Using Neural Network
Similarity: 0.7926
Year: 2022
Source: OpenAlex
URL: https://doi.org/10.3390/s22020572

#3
Title: BlockSim: An Extensible Simulation Tool for Blockchain Systems
Similarity: 0.7832
Year: 2020
Source: arXiv
URL: https://arxiv.org/abs/2004.13438v2

#4
Title: A Survey on Consensus Mechanisms and Mining Strategy Management in Blockchain Networks
Similarity: 0.7805
Year: 2019
Source: OpenAlex
URL: https://doi.org/10.1109/access.2019.2896108

#5
Title: The adoption of cryptocurrency as a disruptive force: Deep learning-based dual stage structural equation modelling and artificial neural network analysis
Similarity: 0.7746
Year: 2021
Source: OpenAlex
URL: https://doi.org/10.1371/journal.pone.0247582

#6
Title: A s

In [84]:
def rerank_external_papers(
    source_text: str,
    papers: list[dict],
    candidate_k: int = 20,
    top_k: int = 10
):
    """
    Rerank externally discovered papers using
    a cross-encoder.
    """

    candidates = papers[:candidate_k]

    pairs = [
        (
            source_text,
            build_external_paper_text(paper)
        )
        for paper in candidates
    ]

    scores = reranker.predict(pairs)

    ranked = []

    for paper, score in zip(
        candidates,
        scores
    ):
        result = paper.copy()

        result["rerank_score"] = float(score)

        ranked.append(result)

    ranked.sort(
        key=lambda paper: paper["rerank_score"],
        reverse=True
    )

    return ranked[:top_k]

In [85]:
similar_papers = rerank_external_papers(
    source_text=source_paper_text,
    papers=rankable_papers,
    candidate_k=20,
    top_k=10
)

print(
    "Final similar papers:",
    len(similar_papers)
)

Final similar papers: 10


In [86]:
for i, paper in enumerate(
    similar_papers,
    start=1
):

    print("\n" + "=" * 100)

    print(f"#{i}")
    print("Title:", paper["title"])
    print(
        "Authors:",
        ", ".join(paper["authors"][:5])
    )
    print("Year:", paper["year"])
    print("Source:", paper["source"])

    print(
        "Embedding similarity:",
        round(
            paper["embedding_similarity"],
            4
        )
    )

    print(
        "Reranker score:",
        round(
            paper["rerank_score"],
            4
        )
    )

    print("URL:", paper["url"])


#1
Title: A Survey on Consensus Mechanisms and Mining Strategy Management in Blockchain Networks
Authors: Wenbo Wang, Dinh Thai Hoang, Peizhao Hu, Zehui Xiong, Dusit Niyato
Year: 2019
Source: OpenAlex
Embedding similarity: 0.7805
Reranker score: -1.7041
URL: https://doi.org/10.1109/access.2019.2896108

#2
Title: A survey on blockchain technology and its security
Authors: Huaqun Guo, Xingjie Yu
Year: 2022
Source: OpenAlex
Embedding similarity: 0.7733
Reranker score: -1.914
URL: https://doi.org/10.1016/j.bcra.2022.100067

#3
Title: A systematic literature review of blockchain-based applications: Current status, classification and open issues
Authors: Fran Casino, Thomas K. Dasaklis, Constantinos Patsakis
Year: 2018
Source: OpenAlex
Embedding similarity: 0.7648
Reranker score: -1.9954
URL: https://doi.org/10.1016/j.tele.2018.11.006

#4
Title: ChainSplitter: Towards Blockchain-based Industrial IoT Architecture for Supporting Hierarchical Storage
Authors: Gang Wang, Zhijie Jerry Shi, Mark 

In [87]:
def explain_paper_relationship(
    source_paper: str,
    related_paper: dict
) -> str:

    prompt = f"""
You are a research analysis assistant.

Compare the source research paper with the related paper.

SOURCE PAPER
============
{source_paper[:7000]}

RELATED PAPER
=============
Title: {related_paper["title"]}

Authors:
{", ".join(related_paper.get("authors", [])[:10])}

Abstract:
{related_paper.get("abstract", "")[:7000]}

Analyze ONLY information supported by the provided text.

Explain:

1. Shared research problem
2. Shared methodology or algorithms
3. Shared application/domain
4. Important differences
5. Why the related paper is relevant to the source paper

Do not invent datasets, results, algorithms, or conclusions.

Return the analysis in this format:

Shared Problem:
...

Shared Methods:
...

Shared Domain:
...

Key Differences:
...

Why It Is Relevant:
...
"""

    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    return response.choices[0].message.content

In [88]:
top_paper = similar_papers[0]

explanation = explain_paper_relationship(
    source_paper=source_paper_text,
    related_paper=top_paper
)

print("RELATED PAPER")
print("=" * 100)
print(top_paper["title"])

print("\nANALYSIS")
print("=" * 100)
print(explanation)

RELATED PAPER
A Survey on Consensus Mechanisms and Mining Strategy Management in Blockchain Networks

ANALYSIS
Shared Problem:
The shared research problem between the source paper and the related paper is the exploration and improvement of blockchain technology. The source paper focuses on blockchain wallet risk prediction using graph neural networks, while the related paper surveys consensus mechanisms and mining strategy management in blockchain networks. Both papers aim to contribute to the development and security of blockchain systems.

Shared Methods:
There is no explicit mention of shared methodologies or algorithms between the two papers. The source paper uses graph neural networks for risk prediction, while the related paper discusses various consensus mechanisms and their impact on blockchain networks.

Shared Domain:
The shared application/domain between the two papers is blockchain technology. Both papers deal with different aspects of blockchain systems, with the source pa

In [89]:
for i, paper in enumerate(similar_papers[:3], start=1):

    print("\n" + "#" * 100)
    print(f"RELATED PAPER #{i}")
    print("#" * 100)

    print("Title:", paper["title"])
    print("Year:", paper["year"])
    print("Source:", paper["source"])
    print("URL:", paper["url"])

    explanation = explain_paper_relationship(
        source_paper=source_paper_text,
        related_paper=paper
    )

    print("\n" + explanation)


####################################################################################################
RELATED PAPER #1
####################################################################################################
Title: A Survey on Consensus Mechanisms and Mining Strategy Management in Blockchain Networks
Year: 2019
Source: OpenAlex
URL: https://doi.org/10.1109/access.2019.2896108

Shared Problem:
The shared research problem between the source paper and the related paper is the exploration of blockchain technology, although the source paper focuses on blockchain wallet risk prediction using graph neural networks, while the related paper delves into consensus mechanisms and mining strategy management in blockchain networks.

Shared Methods:
There is no explicit mention of shared methodologies or algorithms between the two papers. The source paper discusses the use of graph neural networks for risk prediction, whereas the related paper surveys consensus mechanisms and mining strat

In [90]:
def discover_similar_research(
    document: ResearchDocument,
    num_queries: int = 3,
    results_per_query: int = 5,
    candidate_k: int = 20,
    top_k: int = 10,
    explain_top: int = 3
):
    """
    Complete external research discovery pipeline.

    PDF
      ↓
    Research profile
      ↓
    Groq query generation
      ↓
    OpenAlex + arXiv
      ↓
    Deduplication
      ↓
    BGE semantic ranking
      ↓
    Cross-encoder reranking
      ↓
    Groq relationship analysis
    """

    # --------------------------------------------------
    # 1. Build research profile
    # --------------------------------------------------

    profile = build_paper_profile(document)

    # --------------------------------------------------
    # 2. Generate scholarly queries
    # --------------------------------------------------

    queries = generate_research_queries(
        profile,
        num_queries=num_queries
    )

    # --------------------------------------------------
    # 3. Search external research
    # --------------------------------------------------

    candidates = discover_research(
        queries,
        results_per_query=results_per_query
    )

    # --------------------------------------------------
    # 4. Deduplicate
    # --------------------------------------------------

    candidates = deduplicate_papers(
        candidates
    )

    # --------------------------------------------------
    # 5. Keep papers with abstracts
    # --------------------------------------------------

    rankable = [
        paper
        for paper in candidates
        if paper.get("title")
        and paper.get("abstract")
        and len(
            paper["abstract"].strip()
        ) > 50
    ]

    if not rankable:
        return {
            "queries": queries,
            "papers": []
        }

    # --------------------------------------------------
    # 6. Source embedding
    # --------------------------------------------------

    source_embedding = embed_text(
        profile[:12000]
    )

    # --------------------------------------------------
    # 7. Candidate embeddings
    # --------------------------------------------------

    candidate_texts = [
        build_external_paper_text(paper)
        for paper in rankable
    ]

    candidate_embeddings = embed_texts(
        candidate_texts
    )

    # --------------------------------------------------
    # 8. Semantic similarity
    # --------------------------------------------------

    scores = (
        candidate_embeddings
        @ source_embedding
    )

    for paper, score in zip(
        rankable,
        scores
    ):
        paper["embedding_similarity"] = float(
            score
        )

    rankable.sort(
        key=lambda x: x["embedding_similarity"],
        reverse=True
    )

    # --------------------------------------------------
    # 9. Cross-encoder reranking
    # --------------------------------------------------

    final_papers = rerank_external_papers(
        source_text=profile[:12000],
        papers=rankable,
        candidate_k=candidate_k,
        top_k=top_k
    )

    # --------------------------------------------------
    # 10. Explain top papers
    # --------------------------------------------------

    for paper in final_papers[:explain_top]:

        paper["relationship_analysis"] = (
            explain_paper_relationship(
                source_paper=profile,
                related_paper=paper
            )
        )

    return {
        "queries": queries,
        "papers": final_papers
    }

In [91]:
research_discovery = discover_similar_research(
    document
)


Searching: Graph Neural Network for blockchain risk assessment

Searching: Blockchain wallet risk prediction using machine learning algorithms

Searching: Deep learning approaches for cryptocurrency wallet security analysis


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [92]:
print("SEARCH QUERIES")
print("=" * 100)

for query in research_discovery["queries"]:
    print("-", query)

SEARCH QUERIES
- Graph Neural Network for blockchain risk assessment
- Blockchain wallet risk prediction using machine learning algorithms
- Deep learning approaches for cryptocurrency wallet security analysis


In [93]:
print("\nRELATED RESEARCH")
print("=" * 100)

for i, paper in enumerate(
    research_discovery["papers"],
    start=1
):

    print(f"\n#{i}")
    print("Title:", paper["title"])
    print("Year:", paper["year"])
    print("Source:", paper["source"])

    print(
        "Embedding similarity:",
        round(
            paper["embedding_similarity"],
            4
        )
    )

    print(
        "Reranker score:",
        round(
            paper["rerank_score"],
            4
        )
    )

    print("Link:", paper["url"])


RELATED RESEARCH

#1
Title: A Survey on Consensus Mechanisms and Mining Strategy Management in Blockchain Networks
Year: 2019
Source: OpenAlex
Embedding similarity: 0.7805
Reranker score: -1.7041
Link: https://doi.org/10.1109/access.2019.2896108

#2
Title: A survey on blockchain technology and its security
Year: 2022
Source: OpenAlex
Embedding similarity: 0.7733
Reranker score: -1.914
Link: https://doi.org/10.1016/j.bcra.2022.100067

#3
Title: A systematic literature review of blockchain-based applications: Current status, classification and open issues
Year: 2018
Source: OpenAlex
Embedding similarity: 0.7648
Reranker score: -1.9954
Link: https://doi.org/10.1016/j.tele.2018.11.006

#4
Title: ChainSplitter: Towards Blockchain-based Industrial IoT Architecture for Supporting Hierarchical Storage
Year: 2019
Source: arXiv
Embedding similarity: 0.7227
Reranker score: -2.2999
Link: https://arxiv.org/abs/1910.00742v1

#5
Title: Blockchain Security Attacks, Challenges, and Solutions for the F

In [94]:
print("\nRESEARCH RELATIONSHIPS")
print("=" * 100)

for paper in research_discovery["papers"][:3]:

    print("\n" + "-" * 100)
    print(paper["title"])
    print("-" * 100)

    print(
        paper["relationship_analysis"]
    )


RESEARCH RELATIONSHIPS

----------------------------------------------------------------------------------------------------
A Survey on Consensus Mechanisms and Mining Strategy Management in Blockchain Networks
----------------------------------------------------------------------------------------------------
Shared Problem:
The shared research problem between the source paper and the related paper is the exploration of blockchain technology, although the source paper focuses on blockchain wallet risk prediction using graph neural networks, while the related paper delves into consensus mechanisms and mining strategy management in blockchain networks.

Shared Methods:
There is no explicit mention of shared methodologies or algorithms between the two papers. The source paper discusses the use of graph neural networks for risk prediction, whereas the related paper surveys consensus mechanisms and mining strategies without specifying a particular algorithm used in the source paper.

Sha

In [110]:
queries = [
    "What model architecture was used for blockchain wallet risk prediction?",
    "What dataset was used for blockchain wallet risk prediction?",
    "What features were used for the wallet risk prediction model?",
    "How was the model evaluated?"
]

for query in queries:

    print("\n" + "=" * 100)
    print("QUERY:", query)
    print("=" * 100)

    results = retrieve(
        query=query,
        candidate_k=10,
        top_k=5
    )

    for rank, result in enumerate(results, start=1):

        print(
            f"\nRank {rank}"
            f" | Chunk: {result['chunk_id']}"
            f" | Page: {result['page']}"
            f" | Score: {result.get('rerank_score', 'N/A')}"
        )

        print(result["text"][:700])


QUERY: What model architecture was used for blockchain wallet risk prediction?

Rank 1 | Chunk: 24 | Page: 18 | Score: 6.55225944519043
3. System Design
3.1 Environment Setup
We have used the Google Colab environment for the development of
the Blockchain Wallet Risk Prediction system, with datasets and model
checkpoints stored on Google Drive. All the required libraries are installed.
• numpy
• pandas
• torch
• torch.nn / torch.nn.functional
• torch geometric.data (Data)
• torch geometric.loader (NeighborLoader)
• torch geometric.nn (SAGEConv, GCNConv, GATConv)
• torch geometric.utils (coalesce)
• sklearn.model selection
• sklearn.preprocessing (StandardScaler)
• sklearn.metrics
• matplotlib.pyplot
• os, random, warnings, shutil
3.2 Dataset Description
The dataset used in this project consists of Bitcoin wallet-level feature

Rank 2 | Chunk: 48 | Page: 38 | Score: 5.910276412963867
CHAPTER 7
CONCLUSION
• This project presents a Graph Neural Network based framework for
blockchain walle

In [111]:
import math


evaluation_dataset = [
    {
        "query": "What model architecture was used for blockchain wallet risk prediction?",
        "relevant_chunks": [24, 48]
    },

    # We will fill these after inspecting the results
    {
        "query": "What dataset was used for blockchain wallet risk prediction?",
        "relevant_chunks": []
    },

    {
        "query": "What features were used for the wallet risk prediction model?",
        "relevant_chunks": []
    },

    {
        "query": "How was the model evaluated?",
        "relevant_chunks": []
    }
]


def recall_at_k(results, relevant_chunks, k=5):

    retrieved_chunks = {
        result["chunk_id"]
        for result in results[:k]
    }

    relevant_chunks = set(relevant_chunks)

    if not relevant_chunks:
        return 0.0

    return len(
        retrieved_chunks & relevant_chunks
    ) / len(relevant_chunks)


def reciprocal_rank(results, relevant_chunks):

    relevant_chunks = set(relevant_chunks)

    for rank, result in enumerate(
        results,
        start=1
    ):

        if result["chunk_id"] in relevant_chunks:
            return 1.0 / rank

    return 0.0


def ndcg_at_k(results, relevant_chunks, k=5):

    relevant_chunks = set(relevant_chunks)

    dcg = 0.0

    for rank, result in enumerate(
        results[:k],
        start=1
    ):

        relevance = (
            1
            if result["chunk_id"] in relevant_chunks
            else 0
        )

        dcg += (
            relevance /
            math.log2(rank + 1)
        )

    ideal_relevant = min(
        len(relevant_chunks),
        k
    )

    if ideal_relevant == 0:
        return 0.0

    idcg = sum(
        1 / math.log2(rank + 1)
        for rank in range(
            1,
            ideal_relevant + 1
        )
    )

    return dcg / idcg


def evaluate_retrieval(
    evaluation_dataset,
    candidate_k=10,
    top_k=5
):

    recalls = []
    reciprocal_ranks = []
    ndcg_scores = []

    valid_queries = 0

    for item in evaluation_dataset:

        # Don't evaluate queries whose ground truth
        # has not yet been established.
        if not item["relevant_chunks"]:
            continue

        valid_queries += 1

        results = retrieve(
            query=item["query"],
            candidate_k=candidate_k,
            top_k=top_k
        )

        recalls.append(
            recall_at_k(
                results,
                item["relevant_chunks"],
                k=top_k
            )
        )

        reciprocal_ranks.append(
            reciprocal_rank(
                results,
                item["relevant_chunks"]
            )
        )

        ndcg_scores.append(
            ndcg_at_k(
                results,
                item["relevant_chunks"],
                k=top_k
            )
        )

    if valid_queries == 0:
        return {
            "Recall@5": 0.0,
            "MRR": 0.0,
            "NDCG@5": 0.0
        }

    return {
        "Recall@5": sum(recalls) / len(recalls),
        "MRR": sum(reciprocal_ranks) / len(reciprocal_ranks),
        "NDCG@5": sum(ndcg_scores) / len(ndcg_scores)
    }

In [112]:
metrics = evaluate_retrieval(
    evaluation_dataset
)

print("Retrieval Evaluation")
print("=" * 50)

for metric, value in metrics.items():
    print(f"{metric}: {value:.4f}")

Retrieval Evaluation
Recall@5: 1.0000
MRR: 1.0000
NDCG@5: 1.0000
